<a href="https://colab.research.google.com/github/NahomKidane/ai301-labs/blob/main/m04-classification/guided-labs/AI301_Guided_Lab_Part3_Library_Classification_With_Generative_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 301 Guided Lab: Classification, Part 3 · Classification With Generative Models

**Module 4 · Classification**

> **File > Save a copy in Drive** before you start. Switch the runtime on: **Runtime > Change
> runtime type > T4 GPU**. This notebook downloads two models, one of roughly 300 MB and one of
> roughly 1 GB, and one dataset of roughly 1 MB.

Parts 1 and 2 used models that return a label or a vector. This notebook uses models that return
text. To classify a review we write an instruction, the model generates an answer in words, and we
convert that answer into a label.

Two models, both downloaded and run in this notebook:

- **Flan-T5**, an encoder decoder model trained to follow short instructions.
- **Qwen2.5 Instruct**, a chat style decoder model.

No API key is needed for either. An optional extension at the end shows how to run the same task
against a hosted model on a free tier, if you want to try it.

In [ ]:
!pip install -q transformers datasets

---

## Part 0 · Setup

This is a separate notebook, so the dataset and the evaluation helper are loaded again. Nothing
here is new.

In [ ]:
# quiet the loading reports and the progress bar warning, neither is an error
import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

data = load_dataset("cornell-movie-review-data/rotten_tomatoes")

# use the GPU if the runtime has one, otherwise the CPU
device = 0 if torch.cuda.is_available() else -1
print("running on", "GPU" if device == 0 else "CPU")


def evaluate(y_true, y_pred, class_names=("negative", "positive"), title=""):
    """Print the per class scores and draw the confusion matrix as counts and as percentages."""
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    counts = confusion_matrix(y_true, y_pred)
    rates = confusion_matrix(y_true, y_pred, normalize="true")

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))

    ConfusionMatrixDisplay(counts, display_labels=class_names).plot(
        ax=axes[0], cmap="Blues", colorbar=False
    )
    axes[0].set_title("Counts")

    ConfusionMatrixDisplay(rates, display_labels=class_names).plot(
        ax=axes[1], cmap="Blues", colorbar=False, values_format=".0%"
    )
    axes[1].set_title("Percent of each true class")

    if title:
        figure.suptitle(title)

    plt.tight_layout()
    plt.show()

---

## Part 1 · Classifying by instruction

A generative model does not have a list of classes. It continues text. To use one as a classifier
we put the question into the text itself.

For each review we build a string:

```
Is the following sentence positive or negative? the rock is destined to be the 21st century's ...
```

The model reads it and generates an answer, such as `positive`. We then convert that word into the
label the dataset uses, 0 or 1.

- The wording of the instruction is part of the method. Change the wording and the predictions
  change.
- The output is text, so it has to be converted into labels, and the model can produce an answer
  the conversion code does not expect.

Flan-T5 is an encoder decoder model trained on a large collection of tasks written as instructions,
which is why it responds to a plain question with a plain answer.

Generating text takes three steps. The tokenizer turns the string into token ids, `generate`
produces new token ids, and the tokenizer turns those back into text. The first cell below does all
three for one review, with nothing hidden.

- **Python note:** `padding=True` makes every string in a batch the same length so they can run
  together. `max_new_tokens=5` stops the model after a few tokens, since the answer is one word.
  `skip_special_tokens=True` drops the markers the model adds around its output.

> Model card: [google/flan-t5-small](https://huggingface.co/google/flan-t5-small). Read what it was
> trained on and what sizes are available, since we are using the smallest.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

if device == 0:
    model = model.to("cuda")

prompt = "Is the following sentence positive or negative? "
review = data["test"]["text"][0]

# one review, the three steps written out
inputs = tokenizer(prompt + review, return_tensors="pt")

if device == 0:
    inputs = inputs.to("cuda")

# max_new_tokens caps how much the model writes. The answer is one word, so five is plenty,
# and a low cap keeps the run fast. Raise it and the model is free to write a sentence.
outputs = model.generate(**inputs, max_new_tokens=5)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(review)
print()
print("token ids in :", inputs["input_ids"].shape)
print("token ids out:", outputs.shape)
print("answer       :", repr(answer))

Those three steps are the same for every review, so the next cell puts them in a function. The
function also runs the model on a group of strings at a time, which is much faster than one call
per review when there are 1,066 of them.

In [ ]:
def generate_answers(texts, batch_size=32):
    """Run the model over a list of strings and return what it generated for each one."""
    answers = []

    # the model runs on a group of strings at a time rather than one by one
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        # step 1, text into token ids.
        # padding=True pads the shorter strings so every row in the batch is the same length,
        # which is what lets them run together.
        # truncation=True cuts anything longer than the model accepts, 512 tokens here.
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)

        if device == 0:
            inputs = inputs.to("cuda")

        # step 2, the model writes new token ids, at most five of them
        outputs = model.generate(**inputs, max_new_tokens=5)

        # step 3, token ids back into text, one answer per input string
        for row in outputs:
            answers.append(tokenizer.decode(row, skip_special_tokens=True))

    return answers


# the same review as above, now through the function
print(generate_answers([prompt + review]))

**Interpret the output.**

1. Write down exactly what the model generated, including its capitalisation.
2. The dataset uses 0 and 1. Say what still has to happen before this answer can be compared with a
   dataset label.

**Your answer:**

_Type here._

---

## Part 2 · What the model actually generates

This cell runs twenty reviews and counts the distinct answers, so you can see what the parsing code
has to handle.

- **Python note:** `Counter` counts how many times each value appears in a list. `most_common()`
  returns those counts, largest first.

> **Predict before running.**
>
> 1. How many distinct strings do you expect from twenty reviews?
> 2. Name one answer a model could give that would not be the word positive or the word negative.

**Your prediction:**

_Type here._

In [ ]:
from collections import Counter

first_twenty = []
for i in range(20):
    first_twenty.append(prompt + data["test"]["text"][i])

answers = generate_answers(first_twenty)

print(Counter(answers).most_common())

**Interpret the output.**

1. List every distinct string the model produced.
2. Say which of them a parser would have to handle, and which would break a parser that only checks
   for the exact word negative.

**Your answer:**

_Type here._

---

## Part 3 · Parsing the answer

The textbook writes the conversion in one line:

```python
y_pred.append(0 if text == "negative" else 1)
```

Anything other than the exact string `negative` becomes positive. That includes a capitalised
`Negative`, a word like `neutral`, or a whole sentence, if the model produces any of them.

Whether it does is a question about this model and this dataset, so the cell below runs all 1,066
reviews, prints every distinct answer that came back, and then applies both conversions so you can
count how many reviews they disagree on.

- **Python note:** `.strip()` removes spaces and newlines from the ends of a string, and `.lower()`
  makes it lowercase, so `"Negative "` and `"negative"` become the same value.

> **Predict before running.**
>
> How many distinct answers do you expect across 1,066 reviews, and on how many reviews will the
> two conversions disagree?

**Your prediction:**

_Type here._

In [ ]:
# tqdm draws the progress bar. Wrap any loop in it and you get a bar with a count and an
# estimate of the time left, which matters when a cell runs for minutes.
from tqdm import tqdm

# build the instruction for every test review first
prompts = []
for review_text in data["test"]["text"]:
    prompts.append(prompt + review_text)

all_answers = []
for start in tqdm(range(0, len(prompts), 32)):
    all_answers.extend(generate_answers(prompts[start:start + 32]))

# every distinct string the model produced, with how often
print(Counter(all_answers).most_common())

In [ ]:
# the textbook's conversion, one line
book_pred = []
for raw_answer in all_answers:
    book_pred.append(0 if raw_answer == "negative" else 1)

# ours, which checks for both words and keeps anything else aside
y_pred = []
unparsed = []

for raw_answer in all_answers:
    answer = raw_answer.strip().lower()

    if answer == "negative":
        y_pred.append(0)
    elif answer == "positive":
        y_pred.append(1)
    else:
        # nothing we recognise, record it and fall back to negative
        unparsed.append(answer)
        y_pred.append(0)

disagreements = 0
for i in range(len(y_pred)):
    if y_pred[i] != book_pred[i]:
        disagreements += 1

print("answers we could not read:", len(unparsed))
print(Counter(unparsed).most_common(5))
print("reviews the two conversions label differently:", disagreements)

In [ ]:
evaluate(data["test"]["label"], y_pred, title="Flan-T5 small")

**Interpret the output.**

1. Look at the distinct answers. Say whether the textbook's one line conversion would have
   mislabelled anything here, and how you know.
2. Look at the confusion matrix. Say whether the model leans towards one class, and how that
   compares with the task specific model lab.
3. Suppose the unparsed count had been 200. Say how you would find out what the model was
   generating and what you would change.

**Your answer:**

_Type here._

---

## Part 4 · A chat style model

Flan-T5 answers a bare question. A chat model expects a conversation, so the instruction is wrapped
in a message with a role, the same shape used by hosted models such as GPT.

We ask for a single character, 0 or 1, rather than a word.

`Qwen2.5-0.5B-Instruct` has 500 million parameters, which is small for a chat model. Running it over
the whole test split is slow, so this section uses the first 200 reviews. The score is therefore
measured on 200 reviews, not 1,066, and is not directly comparable with the numbers above.

- **Python note:** passing a list of messages to a `"text-generation"` pipeline applies the model's
  chat template. The reply is the last message of `generated_text`, so `[-1]["content"]` is the
  text the model produced.

> Model card: [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).

> **Predict before running.**
>
> 1. Will a 0.5 billion parameter chat model beat Flan-T5 small on this task?
> 2. This model was never trained on movie reviews. Say what it is relying on instead.

**Your prediction:**

_Type here._

In [ ]:
chat = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=device,
    max_new_tokens=4,
)

instruction = (
    "Is the following movie review positive or negative? "
    "Answer with 1 for positive and 0 for negative. Give no other text.\n\n"
)

sample_size = 200

chat_pred = []
chat_unparsed = []

for i in tqdm(range(sample_size)):
    review = data["test"]["text"][i]
    messages = [{"role": "user", "content": instruction + review}]
    reply = chat(messages)[0]["generated_text"][-1]["content"].strip()

    if "1" in reply:
        chat_pred.append(1)
    elif "0" in reply:
        chat_pred.append(0)
    else:
        chat_unparsed.append(reply)
        chat_pred.append(0)

print("answers we could not parse:", len(chat_unparsed))

In [ ]:
evaluate(data["test"]["label"][:sample_size], chat_pred, title="Qwen2.5 0.5B Instruct, first 200 reviews")

**Interpret the output.**

1. Compare this with Flan-T5. Say which did better and name one difference between the two models
   that could explain it.
2. The textbook ran GPT 3.5 on the full test split and reports macro F1 0.91. Say what that model
   has that this one does not.
3. We asked for 0 or 1 rather than a word. Say what that changed about the parser, and one risk it
   introduces.

**Your answer:**

_Type here._

---

## Part 5 · Generative against the earlier routes

Fill this in from your own runs. The first four rows come from the two earlier labs.

| Route | Macro F1 | Labels used | What you have to write |
|---|---|---|---|
| Task specific model | | somebody else's | nothing |
| Embeddings plus logistic regression | | all 8,530 | a few lines of scikit-learn |
| Class averages | | all 8,530 | a few lines of numpy |
| Zero shot with embeddings | | none | two class descriptions |
| Flan-T5 | | none | an instruction |
| Qwen2.5 Instruct, 200 reviews | | none | an instruction and a chat message |
| GPT 3.5, from the textbook | 0.91 | none | an instruction |

> **Your turn.**
>
> Change the wording of the Flan-T5 instruction, for example to
> `"Classify the sentiment of this movie review as positive or negative: "`, rerun Part 3 and
> record the new score. Then say what that tells you about reporting a single number for a
> generative classifier.

In [ ]:
# Your turn: change the instruction and rerun.

---

## Check yourself

- Say what a generative model returns, and what has to happen before it can be scored.
- Name the defect in the one line parser from the textbook.
- Say why two people can get different scores from the same model on the same dataset.
- Give one reason to use a generative model for classification, and one reason not to.

## Summary

A generative model classifies by following an instruction. The instruction is part of the method,
so changing its wording changes the result.

Code that treats every unexpected answer as one class reports a score without reporting that it
guessed, which is why we counted the answers we could not read.

Nothing here was trained. The labelled training set was not used at all, which puts these routes
next to zero shot rather than next to the classifiers in Part 2.

The two models we ran differ in size and in training. In the textbook's runs, a hosted model
scored higher than both of these.

## Optional Extension A · A hosted model through a free API

The textbook runs GPT 3.5 through the OpenAI API, which needs a paid key. This extension runs the
same task against a provider with a free tier, so you can compare a hosted model with the two you
ran above. It is optional and nothing above it depends on it.

Two providers issue a key with an email address and no credit card. Both accept the OpenAI client
library, so the only things that change are the three variables at the top of the cell.

| | Groq | Google AI Studio |
|---|---|---|
| Key from | [console.groq.com](https://console.groq.com) | [ai.google.dev/gemini-api/docs/api-key](https://ai.google.dev/gemini-api/docs/api-key) |
| Base URL | `https://api.groq.com/openai/v1` | `https://generativelanguage.googleapis.com/v1beta/openai/` |
| A model to try | `openai/gpt-oss-20b` | `gemini-2.0-flash` |
| Free limits | 1,000 requests a day | shown per project in the console |
| Note | serves open weight models from OpenAI, Meta, Google and Alibaba | free tier prompts may be used for training |

Model names change, so check the provider's own model list if a name is rejected.

### Using the API safely

An API key is a password that spends your quota. Anyone who has it can use your account until you
revoke it.

1. **Keep the key out of the notebook.** In Colab, open the key icon in the left sidebar, add a
   secret named `API_KEY`, and switch on notebook access. The cell reads it with
   `userdata.get("API_KEY")`. A key typed into a cell is saved inside the `.ipynb` and travels with
   every copy of it.
2. **Revoke a key that leaks.** If it reaches a notebook, a screenshot or a repository, create a new
   one in the provider's console. Deleting the cell does not undo it.
3. **Test on one call before looping.** Print the first reply, check it is what you expected, then
   run the hundred. A loop with a mistake in it can spend a day's allowance in a minute.
4. **Read the error.** 401 means the key is missing or wrong. 429 means you hit the rate limit, so
   wait. 404 on the model usually means that name was retired.

Free tiers are rate limited, so run a subset rather than all 1,066 reviews. The cell below uses 100.

> Read more: [OpenAI, best practices for API key
> safety](https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety) and
> [Colab secrets](https://colab.research.google.com/notebooks/snippets/secrets.ipynb).

- **Python note:** both providers accept the OpenAI client once `base_url` points at them, so the
  same code works against either one.

In [ ]:
# Optional. Skip this cell if you did not create a key.
!pip install -q openai

from google.colab import userdata
from openai import OpenAI

# the three lines that change between providers
base_url = "https://api.groq.com/openai/v1"
hosted_model = "openai/gpt-oss-20b"
key_name = "API_KEY"

client = OpenAI(api_key=userdata.get(key_name), base_url=base_url)

# check one call works before spending quota on a hundred
test_reply = client.chat.completions.create(
    model=hosted_model,
    messages=[{"role": "user", "content": instruction + data["test"]["text"][0]}],
    temperature=0,
    max_tokens=4,
)
print("first reply:", repr(test_reply.choices[0].message.content))

hosted_pred = []
hosted_size = 100

for i in tqdm(range(hosted_size)):
    review = data["test"]["text"][i]
    response = client.chat.completions.create(
        model=hosted_model,
        messages=[{"role": "user", "content": instruction + review}],
        temperature=0,
        max_tokens=4,
    )
    reply = response.choices[0].message.content.strip()

    if "1" in reply:
        hosted_pred.append(1)
    else:
        hosted_pred.append(0)

evaluate(data["test"]["label"][:hosted_size], hosted_pred, title=hosted_model + ", first 100 reviews")

**Interpret the output.**

1. This was measured on 100 reviews and the earlier numbers on 1,066. Say why that makes the
   comparison weaker, and what you would do about it.
2. The hosted model was not downloaded. Name two things you gave up by calling an API instead of
   running a model yourself.

**Your answer:**

_Type here._

## References

- Alammar and Grootendorst, *Hands-On Large Language Models*, Chapter 4, Text Classification with
  Generative Models. This notebook is adapted from the book's own Chapter 4 notebook.
- The book's Chapter 4 notebook:
  [on GitHub](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)
  and [open in Colab](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)
- Model card: [google/flan-t5-small](https://huggingface.co/google/flan-t5-small)
- Model card: [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
- Dataset card: [cornell-movie-review-data/rotten_tomatoes](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes)
- Chung et al. (2022), [Scaling Instruction-Finetuned Language Models](https://arxiv.org/abs/2210.11416),
  the paper behind the Flan models.
- [transformers pipeline reference](https://huggingface.co/docs/transformers/main_classes/pipelines)